In [ ]:
import pandas as pd
import re
import html
from pathlib import Path

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step1.parquet")
te = pd.read_parquet(SHARED / "test_step1.parquet")

# HTML tag regex — basit ama önceki keşifte 21 tag türünün hepsini kapsıyor
TAG_RE = re.compile(r"<[^>]+>")

# Birden fazla whitespace -> tek boşluk
WS_RE = re.compile(r"\s+")

# Escape karakter sequence'ları (\n, \t, \r ve çift backslash)
ESCAPE_RE = re.compile(r"\\[ntr\\]")

def clean_text(s):
    if not isinstance(s, str) or not s:
        return ""
    # 1. HTML entity'leri çöz: &amp; -> &, &#39; -> '
    s = html.unescape(s)
    # 2. Literal escape sequence'ları boşluğa çevir (\n -> " ")
    s = ESCAPE_RE.sub(" ", s)
    # 3. HTML/MathML tag'lerini at
    s = TAG_RE.sub("", s)
    # 4. Whitespace normalize
    s = WS_RE.sub(" ", s).strip()
    return s

# Uygula
for df in (tr, te):
    df["title_clean"] = df["title"].map(clean_text)
    df["abstract_clean"] = df["abstract"].map(clean_text)

# Doğrulama: temizlik öncesi/sonrası örnekler
print("ÖRNEK TEMİZLEMELER (train'den, HTML içerenler)")
sample = tr[tr["title"].str.contains("<", na=False)].head(3)
for _, r in sample.iterrows():
    print(f"\nÖNCE  : {r['title']}")
    print(f"SONRA : {r['title_clean']}")

# Abstract'larda da bir bak (escape sequence'lı)
print("\n\nABSTRACT ÖRNEK")
abs_sample = tr[tr["abstract"].str.contains(r"\\n", na=False, regex=True)].head(2)
for _, r in abs_sample.iterrows():
    print(f"\nÖNCE  : {r['abstract'][:200]}...")
    print(f"SONRA : {r['abstract_clean'][:200]}...")

# Sayısal kontroller
print("\n\nUZUNLUK KARŞILAŞTIRMASI")
for label, df in [("train", tr), ("test", te)]:
    orig_med = df["abstract"].str.len().median()
    clean_med = df["abstract_clean"].str.len().median()
    diff_pct = (1 - clean_med/orig_med) * 100
    print(f"{label} abstract medyan: {int(orig_med)} -> {int(clean_med)} (% {diff_pct:.1f} azalma)")

# HTML tag kalmış mı? (sıfır olmalı)
remaining_html_tr = tr["title_clean"].str.contains("<", na=False).sum()
remaining_html_te = te["title_clean"].str.contains("<", na=False).sum()
print(f"\nHTML kontrol: train title'da kalan '<' = {remaining_html_tr}, test = {remaining_html_te}")

# Boş hale gelen var mı?
empty_tr = (tr["abstract_clean"].str.len() == 0).sum()
empty_te = (te["abstract_clean"].str.len() == 0).sum()
print(f"Temizlik sonrası boş abstract: train {empty_tr}, test {empty_te}")

# Çok kısa hale gelen (<50 karakter) — flag ekle
tr["short_abstract_flag"] = tr["abstract_clean"].str.len() < 100
te["short_abstract_flag"] = te["abstract_clean"].str.len() < 100
print(f"<100 karakter abstract: train {tr['short_abstract_flag'].sum()}, test {te['short_abstract_flag'].sum()}")

# Kaydet (üstüne yazıyoruz, bir sonraki adımlar üzerinde devam edecek)
tr.to_parquet(SHARED / "train_step2.parquet", index=False)
te.to_parquet(SHARED / "test_step2.parquet", index=False)
print(f"\nKaydedildi: train_step2.parquet {tr.shape}, test_step2.parquet {te.shape}")

ÖRNEK TEMİZLEMELER (train'den, HTML içerenler)

ÖNCE  : Association of Rare <i>CYP39A1</i> Variants With Exfoliation Syndrome Involving the Anterior Chamber of the Eye
SONRA : Association of Rare CYP39A1 Variants With Exfoliation Syndrome Involving the Anterior Chamber of the Eye

ÖNCE  : The ZEB2-dependent EMT transcriptional programme drives therapy resistance by activating nucleotide excision repair genes <i>ERCC1</i> and <i>ERCC4</i> in colorectal cancer
SONRA : The ZEB2-dependent EMT transcriptional programme drives therapy resistance by activating nucleotide excision repair genes ERCC1 and ERCC4 in colorectal cancer

ÖNCE  : Phenotypic continuum of <i>NFU1</i>-related disorders
SONRA : Phenotypic continuum of NFU1-related disorders


ABSTRACT ÖRNEK

ÖNCE  : "Importance\nExfoliation syndrome is a systemic disorder characterized by progressive accumulation of abnormal fibrillar protein aggregates manifesting clinically in the anterior chamber of the eye. T...
SONRA : "Importance Ex

In [3]:
import pandas as pd
from pathlib import Path

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet")

# Title_clean'da hâlâ '<' içeren 3 satırı bul ve göster
problem = tr[tr["title_clean"].str.contains("<", na=False)]
print(f"Problemli satır sayısı: {len(problem)}")
for i, r in problem.iterrows():
    print(f"\n--- Satır idx={i} ---")
    print(f"ORIJ : {r['title']}")
    print(f"CLEAN: {r['title_clean']}")

Problemli satır sayısı: 3

--- Satır idx=2657 ---
ORIJ : Al<sub>x</sub>CoCrFeNi (0.5 < x < 3) high-entropy alloys: from cost-effective synthesis to strengthening via thermomechanical processing
CLEAN: AlxCoCrFeNi (0.5 < x < 3) high-entropy alloys: from cost-effective synthesis to strengthening via thermomechanical processing

--- Satır idx=3786 ---
ORIJ : Distances of galactic radio pulsars; first quadrant:-2° &lt; l &lt; 90° and-2° &lt; <i>b</i> &lt; 2°
CLEAN: Distances of galactic radio pulsars; first quadrant:-2° b < 2°

--- Satır idx=16647 ---
ORIJ : Pre-procedural ultrasound-guided evaluation of left internal jugular vein and brachiocephalic vein: Experience on infants &lt;1 year
CLEAN: Pre-procedural ultrasound-guided evaluation of left internal jugular vein and brachiocephalic vein: Experience on infants <1 year


In [4]:
import pandas as pd
import re
import html
from pathlib import Path

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step1.parquet")
te = pd.read_parquet(SHARED / "test_step1.parquet")

TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")
ESCAPE_RE = re.compile(r"\\[ntr\\]")

def clean_text(s):
    if not isinstance(s, str) or not s:
        return ""
    s = ESCAPE_RE.sub(" ", s)   # \n \t \r at
    s = TAG_RE.sub("", s)        # önce GERÇEK tag'leri at (entity'ler &lt; halinde, dokunmaz)
    s = html.unescape(s)         # sonra entity'leri çöz (matematik <,> artık güvende)
    s = WS_RE.sub(" ", s).strip()
    return s

for df in (tr, te):
    df["title_clean"] = df["title"].map(clean_text)
    df["abstract_clean"] = df["abstract"].map(clean_text)
    df["short_abstract_flag"] = df["abstract_clean"].str.len() < 100

# Doğrulama: 3786 düzelmiş mi?
print("Düzeltme kontrolü (idx 3786):")
print(f"ORIJ : {tr.loc[3786, 'title']}")
print(f"CLEAN: {tr.loc[3786, 'title_clean']}")

# Diğer ikisi hâlâ doğru çalışıyor mu?
for idx in [2657, 16647]:
    print(f"\nidx {idx}:")
    print(f"  ORIJ : {tr.loc[idx, 'title']}")
    print(f"  CLEAN: {tr.loc[idx, 'title_clean']}")

# Final HTML kontrol
remaining_tr = tr["title_clean"].str.contains("<", na=False).sum()
remaining_te = te["title_clean"].str.contains("<", na=False).sum()
print(f"\nKalan '<' karakter: train {remaining_tr}, test {remaining_te}")
print("(Bu sayı meşru matematiksel '<' içeren başlıkları gösterir, sıfır olması gerekmez)")

# Kaydet
tr.to_parquet(SHARED / "train_step2.parquet", index=False)
te.to_parquet(SHARED / "test_step2.parquet", index=False)
print(f"\nKaydedildi: {tr.shape}, {te.shape}")

Düzeltme kontrolü (idx 3786):
ORIJ : Distances of galactic radio pulsars; first quadrant:-2° &lt; l &lt; 90° and-2° &lt; <i>b</i> &lt; 2°
CLEAN: Distances of galactic radio pulsars; first quadrant:-2° < l < 90° and-2° < b < 2°

idx 2657:
  ORIJ : Al<sub>x</sub>CoCrFeNi (0.5 < x < 3) high-entropy alloys: from cost-effective synthesis to strengthening via thermomechanical processing
  CLEAN: AlxCoCrFeNi (0.5 < x < 3) high-entropy alloys: from cost-effective synthesis to strengthening via thermomechanical processing

idx 16647:
  ORIJ : Pre-procedural ultrasound-guided evaluation of left internal jugular vein and brachiocephalic vein: Experience on infants &lt;1 year
  CLEAN: Pre-procedural ultrasound-guided evaluation of left internal jugular vein and brachiocephalic vein: Experience on infants <1 year

Kalan '<' karakter: train 3, test 0
(Bu sayı meşru matematiksel '<' içeren başlıkları gösterir, sıfır olması gerekmez)

Kaydedildi: (21303, 31), (807, 31)


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet")
te = pd.read_parquet(SHARED / "test_step2.parquet")

REF_YEAR = 2024  # Veri setinin en güncel yılı

def add_features(df):
    # main_cat: pipe ile böl, ilk değer
    df["main_cat"] = (
        df["category name"].fillna("UNKNOWN").str.split("|").str[0].str.strip()
    )
    
    # n_subcats: kaç kategori (multi-disipliner sinyali)
    df["n_subcats"] = df["category name"].fillna("").apply(
        lambda x: len([c for c in x.split("|") if c.strip()]) if x else 0
    )
    
    # language_grp: en / tr / other / unknown
    def lang_group(x):
        if pd.isna(x): return "unknown"
        if x == "en": return "en"
        if x == "tr": return "tr"
        return "other"
    df["language_grp"] = df["language"].map(lang_group)
    
    # oa_status: NaN -> unknown
    df["oa_status_clean"] = df["oa_status"].fillna("unknown")
    
    # is_oa: NaN'ı ayrı bir flag yapıyoruz, sonra False olarak da ekliyoruz
    df["is_oa_known"] = df["is_oa"].notna()
    df["is_oa_clean"] = df["is_oa"].fillna(False).astype(bool)
    
    # type: journal-article -> article birleştirme, NaN -> unknown
    type_map = {"journal-article": "article"}
    df["type_clean"] = df["type"].fillna("unknown").replace(type_map)
    
    # is_consortium: authors_count > 50 (NaN olanlar False)
    df["is_consortium"] = (df["authors_count"].fillna(0) > 50).astype(bool)
    
    # years_since_pub: 2024 - publication_year, NaN -> NaN (later imputation)
    df["years_since_pub"] = REF_YEAR - df["publication_year"]
    
    # n_keywords: keywords virgülle ayrılı, NaN -> 0
    df["n_keywords"] = df["keywords"].fillna("").apply(
        lambda x: len([k for k in x.split(",") if k.strip()]) if x else 0
    )
    
    # n_authors_log
    df["n_authors_log"] = np.log1p(df["authors_count"].fillna(0))
    
    # n_refs_log
    df["n_refs_log"] = np.log1p(df["referenced_works_count"].fillna(0))
    
    # abstract_len: temizlenmiş abstract karakter sayısı
    df["abstract_len"] = df["abstract_clean"].fillna("").str.len()
    
    return df

tr = add_features(tr)
te = add_features(te)

# Doğrulama
print("YENİ SÜTUNLAR")
new_cols = ["main_cat", "n_subcats", "language_grp", "oa_status_clean",
            "is_oa_known", "is_oa_clean", "type_clean", "is_consortium",
            "years_since_pub", "n_keywords", "n_authors_log", "n_refs_log",
            "abstract_len"]

print("\n--- main_cat (top 10, train vs test) ---")
a = tr["main_cat"].value_counts().head(10)
b = te["main_cat"].value_counts().head(10)
print(pd.DataFrame({"train": a, "test": b}).fillna(0).astype(int).to_string())

print("\n--- language_grp ---")
for label, df in [("train", tr), ("test", te)]:
    print(f"{label}: {dict(df['language_grp'].value_counts())}")

print("\n--- type_clean ---")
for label, df in [("train", tr), ("test", te)]:
    print(f"{label}: {dict(df['type_clean'].value_counts())}")

print("\n--- is_consortium ---")
for label, df in [("train", tr), ("test", te)]:
    print(f"{label}: True={df['is_consortium'].sum()}, False={(~df['is_consortium']).sum()}")

print("\n--- Sayısal yeni özellikler (train) ---")
print(tr[["n_subcats", "years_since_pub", "n_keywords", 
          "n_authors_log", "n_refs_log", "abstract_len"]].describe().round(2).to_string())

print("\n--- Sayısal yeni özellikler (test) ---")
print(te[["n_subcats", "years_since_pub", "n_keywords",
          "n_authors_log", "n_refs_log", "abstract_len"]].describe().round(2).to_string())

# NaN kontrolü
print("\n--- NaN kontrolü (yeni sütunlar) ---")
for c in new_cols:
    n_tr = tr[c].isna().sum()
    n_te = te[c].isna().sum()
    if n_tr > 0 or n_te > 0:
        print(f"  {c}: train={n_tr}, test={n_te}")
print("(Sadece NaN'lı sütunlar listelendi; years_since_pub'da 113 train + 5 test bekleniyor — enrichment olmayan satırlar)")

# Kaydet
tr.to_parquet(SHARED / "train_step2.parquet", index=False)
te.to_parquet(SHARED / "test_step2.parquet", index=False)
print(f"\nKaydedildi: train {tr.shape}, test {te.shape}")

D:\Projects\Essay\AppData\Local\Temp\ipykernel_35924\3476251359.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_oa_clean"] = df["is_oa"].fillna(False).astype(bool)
D:\Projects\Essay\AppData\Local\Temp\ipykernel_35924\3476251359.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_oa_clean"] = df["is_oa"].fillna(False).astype(bool)


YENİ SÜTUNLAR

--- main_cat (top 10, train vs test) ---
                                           train  test
main_cat                                              
BIOCHEMISTRY & MOLECULAR BIOLOGY             491     0
CARDIAC & CARDIOVASCULAR SYSTEMS               0    23
CHEMISTRY, MULTIDISCIPLINARY                 644     0
CLINICAL NEUROLOGY                           494    30
COMPUTER SCIENCE, ARTIFICIAL INTELLIGENCE      0    22
DENTISTRY, ORAL SURGERY & MEDICINE             0    22
EDUCATION & EDUCATIONAL RESEARCH             578    30
ENGINEERING, ELECTRICAL & ELECTRONIC         350     0
ENGINEERING, MULTIDISCIPLINARY               351     0
MATERIALS SCIENCE, MULTIDISCIPLINARY         349    30
MATHEMATICS                                  454     0
MATHEMATICS, APPLIED                           0    28
MEDICINE, GENERAL & INTERNAL                 815    53
MULTIDISCIPLINARY SCIENCES                   501     0
OBSTETRICS & GYNECOLOGY                        0    18
SURGERY  

 Alt Görev 2.3: TF-IDF Özellikleri

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import pickle
import time

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet").reset_index(drop=True)
te = pd.read_parquet(SHARED / "test_step2.parquet").reset_index(drop=True)

# Index sıralamasını koru ve sabitle (npz ile DataFrame eşleşmesi için kritik)
tr["row_id"] = np.arange(len(tr))
te["row_id"] = np.arange(len(te))

# Birleşik metin: title + abstract
def combine(df):
    title = df["title_clean"].fillna("")
    abst  = df["abstract_clean"].fillna("")
    return (title + " " + abst).str.strip()

train_texts = combine(tr)
test_texts  = combine(te)

print(f"Metin örnekleri (train, ilk satır):\n{train_texts.iloc[0][:300]}...\n")

# TF-IDF vectorizer
vec = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words="english",
    lowercase=True,
    strip_accents="unicode",
)

t0 = time.time()
X_train = vec.fit_transform(train_texts)
fit_time = time.time() - t0

t0 = time.time()
X_test = vec.transform(test_texts)
trans_time = time.time() - t0

print(f"TF-IDF train: {X_train.shape}, fit süre: {fit_time:.1f}s")
print(f"TF-IDF test : {X_test.shape}, transform süre: {trans_time:.1f}s")
print(f"Sparsity (train): {1 - X_train.nnz / (X_train.shape[0]*X_train.shape[1]):.4f}")
print(f"Vocabulary size: {len(vec.vocabulary_)}")

# OOV kontrolü: test setinde TF-IDF tamamen sıfır olan satır var mı?
test_zero_rows = (X_test.sum(axis=1) == 0).sum()
print(f"Test setinde tamamen-sıfır TF-IDF vektörü: {test_zero_rows} (metin train kelime dağarcığıyla hiç örtüşmedi)")

# En yüksek ağırlıklı 20 unigram + 20 bigram (sanity check)
feature_names = np.array(vec.get_feature_names_out())
idf_scores = vec.idf_

# Vocabulary'den unigram ve bigram ayrımı
is_bigram = np.array([" " in f for f in feature_names])
print("\nÖrnek 15 unigram (en düşük IDF = en sık):")
ug_idx = np.argsort(idf_scores[~is_bigram])[:15]
ug_features = feature_names[~is_bigram][ug_idx]
print(", ".join(ug_features))

print("\nÖrnek 15 bigram (en düşük IDF = en sık):")
bg_idx = np.argsort(idf_scores[is_bigram])[:15]
bg_features = feature_names[is_bigram][bg_idx]
print(", ".join(bg_features))

# Kaydet
save_npz(SHARED / "tfidf_train.npz", X_train)
save_npz(SHARED / "tfidf_test.npz", X_test)
with open(SHARED / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vec, f)

# row_id eklenmiş DataFrame'leri tekrar yaz (npz ile eşleşme için)
tr.to_parquet(SHARED / "train_step2.parquet", index=False)
te.to_parquet(SHARED / "test_step2.parquet", index=False)

print(f"\nKaydedildi:")
print(f"  tfidf_vectorizer.pkl ({(SHARED / 'tfidf_vectorizer.pkl').stat().st_size / 1024 / 1024:.1f} MB)")
print(f"  tfidf_train.npz      ({(SHARED / 'tfidf_train.npz').stat().st_size / 1024 / 1024:.1f} MB)")
print(f"  tfidf_test.npz       ({(SHARED / 'tfidf_test.npz').stat().st_size / 1024 / 1024:.1f} MB)")

Metin örnekleri (train, ilk satır):
Association of Rare CYP39A1 Variants With Exfoliation Syndrome Involving the Anterior Chamber of the Eye "Importance Exfoliation syndrome is a systemic disorder characterized by progressive accumulation of abnormal fibrillar protein aggregates manifesting clinically in the anterior chamber of the ey...

TF-IDF train: (21303, 50000), fit süre: 11.6s
TF-IDF test : (807, 50000), transform süre: 0.2s
Sparsity (train): 0.9978
Vocabulary size: 50000
Test setinde tamamen-sıfır TF-IDF vektörü: 0 (metin train kelime dağarcığıyla hiç örtüşmedi)

Örnek 15 unigram (en düşük IDF = en sık):
study, results, using, used, analysis, data, methods, based, patients, different, significant, abstract, high, compared, time

Örnek 15 bigram (en düşük IDF = en sık):
study aimed, covid 19, aim study, study aims, statistically significant, control group, present study, significantly higher, included study, significant difference, mean age, data collected, long term, materials 

In [1]:
print("\nTEMİZLİK ADIMI TAMAMLANDI. Sonraki adımda bu temizlenmiş verilerle çalışacağız.")


TEMİZLİK ADIMI TAMAMLANDI. Sonraki adımda bu temizlenmiş verilerle çalışacağız.


In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
from tqdm import tqdm
import time

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet").reset_index(drop=True)
te = pd.read_parquet(SHARED / "test_step2.parquet").reset_index(drop=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

MODEL_NAME = "allenai/specter2_base"
BATCH_SIZE = 32
MAX_LEN = 512

print(f"Model yükleniyor: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
model.eval()
print(f"Model hazır. Parametre sayısı: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# SPECTER2 input formatı: title + [SEP] + abstract
def build_inputs(df):
    titles = df["title_clean"].fillna("").tolist()
    abstracts = df["abstract_clean"].fillna("").tolist()
    return [t + tokenizer.sep_token + a for t, a in zip(titles, abstracts)]

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx]

def collate(batch):
    return tokenizer(batch, padding=True, truncation=True, 
                     max_length=MAX_LEN, return_tensors="pt")

@torch.no_grad()
def encode(texts, label):
    ds = TextDataset(texts)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, 
                    collate_fn=collate, num_workers=0)
    embeddings = []
    t0 = time.time()
    for batch in tqdm(dl, desc=label):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        # [CLS] token embedding'i
        cls_emb = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_emb.float().cpu().numpy())
    elapsed = time.time() - t0
    arr = np.vstack(embeddings)
    print(f"{label}: shape={arr.shape}, süre={elapsed:.1f}s ({len(texts)/elapsed:.1f} doc/s)")
    return arr

# Train embeddings
train_texts = build_inputs(tr)
print(f"\nÖrnek input (ilk satır, ilk 200 char):\n{train_texts[0][:200]}\n")
emb_train = encode(train_texts, "train")

# Test embeddings
test_texts = build_inputs(te)
emb_test = encode(test_texts, "test")

# Sanity check
print(f"\nNaN kontrol: train {np.isnan(emb_train).any()}, test {np.isnan(emb_test).any()}")
print(f"Boyut: train {emb_train.shape}, test {emb_test.shape}")
print(f"Norm dağılımı (train): mean={np.linalg.norm(emb_train, axis=1).mean():.2f}, "
      f"std={np.linalg.norm(emb_train, axis=1).std():.2f}")

# Sanity: 2 makale arasında benzerlik (rastgele iki tıp makalesi yakın olmalı)
from sklearn.metrics.pairwise import cosine_similarity
# Aynı main_cat'tan iki makale
medicine_idx = tr[tr["main_cat"] == "MEDICINE, GENERAL & INTERNAL"].index[:2].tolist()
math_idx = tr[tr["main_cat"] == "MATHEMATICS"].index[:2].tolist()
if len(medicine_idx) >= 2 and len(math_idx) >= 2:
    sim_within_med = cosine_similarity(
        emb_train[medicine_idx[0]:medicine_idx[0]+1],
        emb_train[medicine_idx[1]:medicine_idx[1]+1])[0, 0]
    sim_med_math = cosine_similarity(
        emb_train[medicine_idx[0]:medicine_idx[0]+1],
        emb_train[math_idx[0]:math_idx[0]+1])[0, 0]
    print(f"\nKosinüs benzerliği sanity check:")
    print(f"  Tıp-Tıp:        {sim_within_med:.3f}")
    print(f"  Tıp-Matematik:  {sim_med_math:.3f}")
    print(f"  (Tıp-Tıp daha yüksek olmalı, aksi halde model işe yaramıyor)")

# Kaydet
np.save(SHARED / "specter2_train.npy", emb_train)
np.save(SHARED / "specter2_test.npy", emb_test)
print(f"\nKaydedildi:")
print(f"  specter2_train.npy ({(SHARED / 'specter2_train.npy').stat().st_size/1e6:.1f} MB)")
print(f"  specter2_test.npy  ({(SHARED / 'specter2_test.npy').stat().st_size/1e6:.1f} MB)")

# GPU temizliği
del model, tokenizer
torch.cuda.empty_cache()

Device: cuda, VRAM: 8.6 GB
Model yükleniyor: allenai/specter2_base


tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

c:\Users\Kerem\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kerem\.cache\huggingface\hub\models--allenai--specter2_base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

train: 100%|██████████| 666/666 [01:53<00:00,  5.85it/s]


train: shape=(21303, 768), süre=113.9s (187.1 doc/s)


test: 100%|██████████| 26/26 [00:04<00:00,  5.92it/s]

test: shape=(807, 768), süre=4.4s (183.7 doc/s)

NaN kontrol: train False, test False
Boyut: train (21303, 768), test (807, 768)
Norm dağılımı (train): mean=21.65, std=0.15

Kosinüs benzerliği sanity check:
  Tıp-Tıp:        0.914
  Tıp-Matematik:  0.782
  (Tıp-Tıp daha yüksek olmalı, aksi halde model işe yaramıyor)

Kaydedildi:
  specter2_train.npy (65.4 MB)
  specter2_test.npy  (2.5 MB)


In [3]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from itertools import combinations
from tqdm import tqdm

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet")
te = pd.read_parquet(SHARED / "test_step2.parquet")

# Train + test'i birleştir (ağ için)
df = pd.concat([tr, te], ignore_index=True)
print(f"Birleşik veri: {df.shape}")

# Konsorsiyum filtresi: authors_count > 50 dışla
n_before = len(df)
df = df[~df["is_consortium"]].copy()
n_excluded = n_before - len(df)
print(f"Konsorsiyum (>50 yazar) filtrelendi: {n_excluded} makale dışlandı, kalan {len(df)}")

# authors sütunu enrichment olmayan satırlarda NaN, dışla
df = df[df["authors"].notna()].copy()
print(f"authors sütunu dolu: {len(df)}")

# JSON parse + author_id listesi çıkar
def parse_author_ids(authors_str):
    """JSON'dan openalex_id listesi çıkar; None olanlar atılır."""
    try:
        arr = json.loads(authors_str)
        return [a["openalex_id"] for a in arr if a.get("openalex_id")]
    except (json.JSONDecodeError, TypeError):
        return []

print("Yazar ID'leri parse ediliyor...")
df["author_ids"] = df["authors"].map(parse_author_ids)
df["n_valid_authors"] = df["author_ids"].map(len)

# Hiç geçerli yazar olmayan veya tek yazar olan makaleleri (çift kuramaz) ayır
multi_author = df[df["n_valid_authors"] >= 2].copy()
print(f"En az 2 geçerli yazarlı makale: {len(multi_author)}")
print(f"Ortalama yazar: {multi_author['n_valid_authors'].mean():.1f}, "
      f"medyan: {multi_author['n_valid_authors'].median():.0f}")

# 1) AUTHOR_PAIRS: her makale için co-authorship çiftleri
print("\nÇiftler oluşturuluyor...")
pairs_records = []
for _, row in tqdm(multi_author.iterrows(), total=len(multi_author)):
    aids = row["author_ids"]
    year = row["publication_year"]
    doi = row["doi"]
    for a, b in combinations(sorted(set(aids)), 2):  # set(): aynı makalede dup yazar olursa
        pairs_records.append((a, b, doi, year))

pairs_df = pd.DataFrame(pairs_records, columns=["author_a", "author_b", "doi", "year"])
pairs_df["year"] = pairs_df["year"].astype("Int64")  # NaN yıl olabilir
print(f"Toplam çift: {len(pairs_df):,}")
print(f"Tekil çift (aynı 2 yazar farklı makalede tekrar): {pairs_df.groupby(['author_a','author_b']).ngroups:,}")

# 2) AUTHOR_STATS: yazar bazlı özetler
print("\nYazar istatistikleri hesaplanıyor...")
exploded = multi_author[["author_ids", "publication_year", "doi", "cited_by_count"]].copy()
exploded = exploded.explode("author_ids").rename(columns={"author_ids": "author_id"})

author_stats = exploded.groupby("author_id").agg(
    n_papers=("doi", "count"),
    first_year=("publication_year", "min"),
    last_year=("publication_year", "max"),
    total_citations=("cited_by_count", "sum"),
    mean_citations=("cited_by_count", "mean"),
).reset_index()
print(f"Tekil yazar sayısı: {len(author_stats):,}")
print(f"Yazar başına ortalama makale: {author_stats['n_papers'].mean():.2f}, "
      f"medyan: {author_stats['n_papers'].median():.0f}, "
      f"max: {author_stats['n_papers'].max()}")

# Derece (kaç farklı co-author): pairs üzerinden
deg_a = pairs_df.groupby("author_a").size().rename("deg_from_a")
deg_b = pairs_df.groupby("author_b").size().rename("deg_from_b")
unique_coauth_a = pairs_df.groupby("author_a")["author_b"].nunique().rename("uniq_coauth_a")
unique_coauth_b = pairs_df.groupby("author_b")["author_a"].nunique().rename("uniq_coauth_b")
deg_total = pd.concat([deg_a, deg_b], axis=1).fillna(0).sum(axis=1).rename("total_pair_count")
uniq_total = pd.concat([unique_coauth_a, unique_coauth_b], axis=1).fillna(0).sum(axis=1).rename("degree")
deg_df = pd.concat([deg_total, uniq_total], axis=1).reset_index().rename(columns={"index": "author_id"})

author_stats = author_stats.merge(deg_df, on="author_id", how="left").fillna({"degree": 0, "total_pair_count": 0})
author_stats["degree"] = author_stats["degree"].astype(int)
author_stats["total_pair_count"] = author_stats["total_pair_count"].astype(int)

print(f"Derece dağılımı: median={author_stats['degree'].median():.0f}, "
      f"p95={author_stats['degree'].quantile(.95):.0f}, "
      f"max={author_stats['degree'].max()}")

# Kaydet
pairs_df.to_parquet(SHARED / "author_pairs.parquet", index=False)
author_stats.to_parquet(SHARED / "author_stats.parquet", index=False)

print(f"\nKaydedildi:")
print(f"  author_pairs.parquet  ({len(pairs_df):,} satır, "
      f"{(SHARED/'author_pairs.parquet').stat().st_size/1e6:.1f} MB)")
print(f"  author_stats.parquet  ({len(author_stats):,} satır, "
      f"{(SHARED/'author_stats.parquet').stat().st_size/1e6:.1f} MB)")

# Sanity check: en aktif 5 yazar
print("\nEn aktif 5 yazar (n_papers'a göre):")
print(author_stats.nlargest(5, "n_papers")[
    ["author_id", "n_papers", "first_year", "last_year", "degree", "total_citations"]
].to_string(index=False))

Birleşik veri: (22110, 45)
Konsorsiyum (>50 yazar) filtrelendi: 667 makale dışlandı, kalan 21443
authors sütunu dolu: 21325
Yazar ID'leri parse ediliyor...
En az 2 geçerli yazarlı makale: 19046
Ortalama yazar: 5.8, medyan: 4

Çiftler oluşturuluyor...


100%|██████████| 19046/19046 [00:00<00:00, 24874.49it/s]


Toplam çift: 574,720
Tekil çift (aynı 2 yazar farklı makalede tekrar): 466,635

Yazar istatistikleri hesaplanıyor...
Tekil yazar sayısı: 61,128
Yazar başına ortalama makale: 1.81, medyan: 1, max: 292
Derece dağılımı: median=8, p95=47, max=696

Kaydedildi:
  author_pairs.parquet  (574,720 satır, 3.4 MB)
  author_stats.parquet  (61,128 satır, 0.9 MB)

En aktif 5 yazar (n_papers'a göre):
                       author_id  n_papers  first_year  last_year  degree  total_citations
https://openalex.org/A5030756029       292      2018.0     2023.0     605           6998.0
https://openalex.org/A5085047470       133      2019.0     2023.0     301           2154.0
https://openalex.org/A5005566845       125      2020.0     2023.0     696           3619.0
https://openalex.org/A5050370140        81      2019.0     2023.0     255           2768.0
https://openalex.org/A5080943561        79      2020.0     2023.0     172            725.0


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import load_npz, hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import ks_2samp, chi2_contingency
import warnings
warnings.filterwarnings("ignore")

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet").reset_index(drop=True)
te = pd.read_parquet(SHARED / "test_step2.parquet").reset_index(drop=True)

y_tr = tr["modded_Q_category"].values
y_le = LabelEncoder().fit(y_tr)
y_tr_enc = y_le.transform(y_tr)
print(f"Hedef sınıflar: {list(y_le.classes_)}")
print(f"Train Q dağılımı: {dict(pd.Series(y_tr).value_counts())}")

# =====================================================================
# 1) FEATURE SETLERİ TEK TEK CV ile değerlendir
# =====================================================================

# A) TF-IDF
X_tfidf = load_npz(SHARED / "tfidf_train.npz")
print(f"\n[A] TF-IDF: {X_tfidf.shape}")
scores_tfidf = cross_val_score(
    LogisticRegression(max_iter=1000, solver="liblinear"),
    X_tfidf, y_tr_enc, cv=5, scoring="f1_macro", n_jobs=-1
)
print(f"   5-fold CV macro F1: {scores_tfidf.mean():.4f} ± {scores_tfidf.std():.4f}")

# B) SPECTER2
X_sp = np.load(SHARED / "specter2_train.npy")
print(f"\n[B] SPECTER2: {X_sp.shape}")
scores_sp = cross_val_score(
    LogisticRegression(max_iter=1000, solver="lbfgs"),
    X_sp, y_tr_enc, cv=5, scoring="f1_macro", n_jobs=-1
)
print(f"   5-fold CV macro F1: {scores_sp.mean():.4f} ± {scores_sp.std():.4f}")

# C) Yapısal/Kategorik (2.2'de oluşturulan ALLOW listesindeki)
# Modül A için "kurumsuz base" feature seti — university DAHİL DEĞİL, journal/publisher DAHİL DEĞİL
# Sadece içerik-tabanlı/genel sinyaller
struct_num = ["n_subcats", "years_since_pub", "n_keywords",
              "n_authors_log", "n_refs_log", "abstract_len"]
struct_cat = ["main_cat", "language_grp", "type_clean", "oa_status_clean"]
struct_bool = ["is_oa_clean", "is_oa_known", "is_consortium", "short_abstract_flag"]

# NaN'ları doldur (CV için, kalıcı değişiklik değil)
def build_struct(df):
    num = df[struct_num].fillna(df[struct_num].median(numeric_only=True)).values
    cat = pd.get_dummies(df[struct_cat].fillna("unknown"), drop_first=False)
    bool_ = df[struct_bool].astype(int).values
    return num, cat, bool_

num_tr, cat_tr, bool_tr = build_struct(tr)
num_te, cat_te, bool_te = build_struct(te)

# Train/test kategoriklerini hizala (test'te eksik kategori olabilir)
cat_te = cat_te.reindex(columns=cat_tr.columns, fill_value=0)

X_struct_tr = np.hstack([num_tr, cat_tr.values, bool_tr]).astype(float)
X_struct_te = np.hstack([num_te, cat_te.values, bool_te]).astype(float)
print(f"\n[C] Yapısal/kategorik: {X_struct_tr.shape}")

scores_struct = cross_val_score(
    LogisticRegression(max_iter=1000, solver="lbfgs"),
    X_struct_tr, y_tr_enc, cv=5, scoring="f1_macro", n_jobs=-1
)
print(f"   5-fold CV macro F1: {scores_struct.mean():.4f} ± {scores_struct.std():.4f}")

# D) Yazar istatistik (her makale için yazarların ortalama derece, n_papers, vb.)
import json
def parse_author_ids(authors_str):
    try:
        arr = json.loads(authors_str)
        return [a["openalex_id"] for a in arr if a.get("openalex_id")]
    except:
        return []

author_stats = pd.read_parquet(SHARED / "author_stats.parquet").set_index("author_id")

def author_aggregates(df):
    rows = []
    for _, r in df.iterrows():
        ids = parse_author_ids(r["authors"]) if isinstance(r.get("authors"), str) else []
        ids = [i for i in ids if i in author_stats.index]
        if not ids:
            rows.append([0, 0, 0, 0, 0])
            continue
        sub = author_stats.loc[ids]
        rows.append([
            len(ids),
            sub["n_papers"].mean(),
            sub["n_papers"].max(),
            sub["degree"].mean(),
            sub["total_citations"].mean(),
        ])
    return np.array(rows, dtype=float)

X_auth_tr = author_aggregates(tr)
X_auth_te = author_aggregates(te)
print(f"\n[D] Yazar istatistikleri: {X_auth_tr.shape}")
scores_auth = cross_val_score(
    LogisticRegression(max_iter=1000, solver="lbfgs"),
    X_auth_tr, y_tr_enc, cv=5, scoring="f1_macro", n_jobs=-1
)
print(f"   5-fold CV macro F1: {scores_auth.mean():.4f} ± {scores_auth.std():.4f}")

# Özet tablo
print("\n" + "="*70)
print("FEATURE SETİ CV SKORLARI (LR, 5-fold, macro F1)")
print("="*70)
summary = pd.DataFrame([
    {"feature_set": "TF-IDF (50K dim)",       "cv_f1": scores_tfidf.mean(),  "cv_std": scores_tfidf.std()},
    {"feature_set": "SPECTER2 (768 dim)",     "cv_f1": scores_sp.mean(),     "cv_std": scores_sp.std()},
    {"feature_set": "Yapısal/kategorik",      "cv_f1": scores_struct.mean(), "cv_std": scores_struct.std()},
    {"feature_set": "Yazar istatistikleri",   "cv_f1": scores_auth.mean(),   "cv_std": scores_auth.std()},
])
def interpret(f1):
    if f1 < 0.30:  return "zayıf (random'a yakın)"
    if f1 < 0.55:  return "makul, sağlıklı sinyal"
    if f1 < 0.75:  return "güçlü ama gerçekçi"
    if f1 < 0.85:  return "çok güçlü, gözlem"
    if f1 < 0.95:  return "ŞÜPHELİ, sızıntı incele"
    return "KESİN SIZINTI"
summary["yorum"] = summary["cv_f1"].map(interpret)
print(summary.to_string(index=False))

# =====================================================================
# 2) TRAIN-TEST DAĞILIM KAYMASI (KS / chi-square)
# =====================================================================
print("\n" + "="*70)
print("TRAIN ↔ TEST DAĞILIM KAYMASI")
print("="*70)

print("\n--- Sayısal feature'lar (KS testi, p<0.01 belirgin kayma) ---")
for col in struct_num:
    a = pd.to_numeric(tr[col], errors="coerce").dropna()
    b = pd.to_numeric(te[col], errors="coerce").dropna()
    stat, p = ks_2samp(a, b)
    flag = "⚠️" if p < 0.01 else "✓"
    print(f"  {flag}  {col:25s}  KS={stat:.3f}  p={p:.2e}")

print("\n--- Kategorik feature'lar (chi-square, p<0.01 belirgin kayma) ---")
for col in struct_cat:
    counts_tr = tr[col].fillna("unknown").value_counts()
    counts_te = te[col].fillna("unknown").value_counts()
    all_keys = sorted(set(counts_tr.index) | set(counts_te.index))
    table = pd.DataFrame({
        "train": counts_tr.reindex(all_keys, fill_value=0),
        "test":  counts_te.reindex(all_keys, fill_value=0),
    }).T
    # 0-only sütunları at (chi2 patlar)
    table = table.loc[:, table.sum() > 0]
    try:
        chi2, p, _, _ = chi2_contingency(table.values)
        flag = "⚠️" if p < 0.01 else "✓"
        print(f"  {flag}  {col:25s}  chi2={chi2:.1f}  p={p:.2e}")
    except Exception as e:
        print(f"     {col}: {e}")

# SPECTER2 dağılımı: ortalama vektör mesafesi (centroid kayması)
sp_te = np.load(SHARED / "specter2_test.npy")
centroid_tr = X_sp.mean(axis=0)
centroid_te = sp_te.mean(axis=0)
from numpy.linalg import norm
cos = (centroid_tr @ centroid_te) / (norm(centroid_tr)*norm(centroid_te))
print(f"\nSPECTER2 centroid cosine (train↔test): {cos:.4f}  (1.0=aynı dağılım)")

# =====================================================================
# 3) NaN/OOV KONSOLİDE
# =====================================================================
print("\n" + "="*70)
print("NaN / OOV KONSOLİDE TABLO")
print("="*70)
nan_summary = []
for label, x_tr, x_te in [
    ("TF-IDF",      X_tfidf,      load_npz(SHARED / "tfidf_test.npz")),
    ("SPECTER2",    X_sp,         sp_te),
    ("Yapısal",     X_struct_tr,  X_struct_te),
    ("Yazar istat", X_auth_tr,    X_auth_te),
]:
    if hasattr(x_tr, "toarray"):  # sparse
        zero_tr = (x_tr.sum(axis=1) == 0).sum()
        zero_te = (x_te.sum(axis=1) == 0).sum()
        nan_tr = nan_te = 0
    else:
        nan_tr = np.isnan(x_tr).any(axis=1).sum()
        nan_te = np.isnan(x_te).any(axis=1).sum()
        zero_tr = (np.abs(x_tr).sum(axis=1) == 0).sum()
        zero_te = (np.abs(x_te).sum(axis=1) == 0).sum()
    nan_summary.append({
        "set": label, "train_n": x_tr.shape[0], "test_n": x_te.shape[0],
        "tr_nan_rows": nan_tr, "te_nan_rows": nan_te,
        "tr_zero_rows": zero_tr, "te_zero_rows": zero_te,
    })
print(pd.DataFrame(nan_summary).to_string(index=False))

Hedef sınıflar: ['Q1', 'Q2', 'Q3', 'Q4']
Train Q dağılımı: {'Q4': 5560, 'Q2': 5552, 'Q1': 5365, 'Q3': 4826}

[A] TF-IDF: (21303, 50000)
   5-fold CV macro F1: 0.3756 ± 0.0195

[B] SPECTER2: (21303, 768)
   5-fold CV macro F1: 0.3735 ± 0.0178

[C] Yapısal/kategorik: (21303, 269)
   5-fold CV macro F1: 0.3657 ± 0.0549

[D] Yazar istatistikleri: (21303, 5)
   5-fold CV macro F1: 0.2955 ± 0.0181

FEATURE SETİ CV SKORLARI (LR, 5-fold, macro F1)
         feature_set    cv_f1   cv_std                  yorum
    TF-IDF (50K dim) 0.375560 0.019495 makul, sağlıklı sinyal
  SPECTER2 (768 dim) 0.373526 0.017765 makul, sağlıklı sinyal
   Yapısal/kategorik 0.365655 0.054928 makul, sağlıklı sinyal
Yazar istatistikleri 0.295469 0.018090 zayıf (random'a yakın)

TRAIN ↔ TEST DAĞILIM KAYMASI

--- Sayısal feature'lar (KS testi, p<0.01 belirgin kayma) ---
  ✓  n_subcats                  KS=0.032  p=3.81e-01
  ✓  years_since_pub            KS=0.046  p=7.30e-02
  ⚠️  n_keywords                 KS=0.083  p=3.

In [5]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
from scipy.sparse import load_npz, hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

SHARED = Path(r"D:\Projects\Essay\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step2.parquet").reset_index(drop=True)
te = pd.read_parquet(SHARED / "test_step2.parquet").reset_index(drop=True)

# Hedef
y_le = LabelEncoder().fit(tr["modded_Q_category"])
y_tr = y_le.transform(tr["modded_Q_category"])
y_te = y_le.transform(te["modded_Q_category"])

# ======================================================================
# 1) FEATURE SETLERİNİ TOPLA
# ======================================================================
# TF-IDF
X_tfidf_tr = load_npz(SHARED / "tfidf_train.npz")
X_tfidf_te = load_npz(SHARED / "tfidf_test.npz")

# SPECTER2
X_sp_tr = np.load(SHARED / "specter2_train.npy")
X_sp_te = np.load(SHARED / "specter2_test.npy")

# Yapısal — base sürüm (university YOK)
struct_num = ["n_subcats", "years_since_pub", "n_keywords",
              "n_authors_log", "n_refs_log", "abstract_len"]
struct_cat = ["main_cat", "language_grp", "type_clean", "oa_status_clean"]
struct_bool = ["is_oa_clean", "is_oa_known", "is_consortium", "short_abstract_flag"]

# Imputation: train medyan ile doldur (test'e aynı medyanı uygula = sızıntı yok)
num_medians = tr[struct_num].median(numeric_only=True)
def build_struct_base(df):
    num = df[struct_num].fillna(num_medians).values.astype(float)
    cat = pd.get_dummies(df[struct_cat].fillna("unknown"), drop_first=False)
    bool_ = df[struct_bool].astype(int).values
    return num, cat, bool_

num_tr, cat_tr, bool_tr = build_struct_base(tr)
num_te, cat_te, bool_te = build_struct_base(te)
# Hizala (test'te eksik kategori olabilir, tersi olamaz çünkü main_cat OOD %100 kapsama)
cat_te = cat_te.reindex(columns=cat_tr.columns, fill_value=0)

X_struct_base_tr = np.hstack([num_tr, cat_tr.values.astype(float), bool_tr.astype(float)])
X_struct_base_te = np.hstack([num_te, cat_te.values.astype(float), bool_te.astype(float)])
struct_feature_names = (
    list(struct_num) + list(cat_tr.columns) + list(struct_bool)
)
print(f"Yapısal (base, university YOK): {X_struct_base_tr.shape}")

# Yapısal — ood sürüm (university freq encoding eklenmiş)
# Train'de her university için ortalama Q1/Q2/Q3/Q4 oranı + n_papers
univ_stats = tr.groupby("university")["modded_Q_category"].value_counts(normalize=True).unstack(fill_value=0)
univ_stats.columns = [f"univ_q{i+1}_rate" for i in range(univ_stats.shape[1])]
univ_papers = tr.groupby("university").size().rename("univ_n_papers_log")
univ_papers = np.log1p(univ_papers)
univ_lookup = univ_stats.join(univ_papers)
print(f"Univ encoding sütunları: {list(univ_lookup.columns)}")
print(f"Bilinen üniversite sayısı: {len(univ_lookup)}")

# Train ortalaması ile doldur (Bahçeşehir için)
global_fill = univ_lookup.mean()
print(f"Global fill (Bahçeşehir gibi unseen için): {global_fill.to_dict()}")

def encode_univ(df):
    rows = df["university"].map(lambda u: univ_lookup.loc[u].values if u in univ_lookup.index else global_fill.values)
    return np.vstack(rows.values).astype(float)

X_univ_tr = encode_univ(tr)
X_univ_te = encode_univ(te)
print(f"Univ encoded: train {X_univ_tr.shape}, test {X_univ_te.shape}")
print(f"Test (Bahçeşehir, hep aynı değer): {X_univ_te[0]}")

X_struct_ood_tr = np.hstack([X_struct_base_tr, X_univ_tr])
X_struct_ood_te = np.hstack([X_struct_base_te, X_univ_te])
struct_ood_names = struct_feature_names + list(univ_lookup.columns)

# Yazar istatistik (2.6'dan tekrar — saklamak için)
author_stats_df = pd.read_parquet(SHARED / "author_stats.parquet").set_index("author_id")
def parse_author_ids(authors_str):
    try:
        arr = json.loads(authors_str)
        return [a["openalex_id"] for a in arr if a.get("openalex_id")]
    except:
        return []
def author_aggregates(df):
    rows = []
    for _, r in df.iterrows():
        ids = parse_author_ids(r["authors"]) if isinstance(r.get("authors"), str) else []
        ids = [i for i in ids if i in author_stats_df.index]
        if not ids:
            rows.append([0, 0, 0, 0, 0])
            continue
        sub = author_stats_df.loc[ids]
        rows.append([len(ids), sub["n_papers"].mean(), sub["n_papers"].max(),
                     sub["degree"].mean(), sub["total_citations"].mean()])
    return np.array(rows, dtype=float)

X_auth_tr = author_aggregates(tr)
X_auth_te = author_aggregates(te)
auth_feature_names = ["n_valid_authors", "auth_mean_papers", "auth_max_papers",
                      "auth_mean_degree", "auth_mean_citations"]

# ======================================================================
# 2) PKL DOSYALARINI YAZ
# ======================================================================
def make_bundle(struct_tr, struct_te, struct_names):
    return {
        "y_train": y_tr, "y_test": y_te,
        "tfidf_train": X_tfidf_tr, "tfidf_test": X_tfidf_te,
        "specter2_train": X_sp_tr, "specter2_test": X_sp_te,
        "struct_train": struct_tr, "struct_test": struct_te,
        "struct_feature_names": struct_names,
        "author_train": X_auth_tr, "author_test": X_auth_te,
        "author_feature_names": auth_feature_names,
        "y_encoder": y_le,
        "doi_train": tr["doi"].values, "doi_test": te["doi"].values,
    }

base_bundle = make_bundle(X_struct_base_tr, X_struct_base_te, struct_feature_names)
ood_bundle  = make_bundle(X_struct_ood_tr,  X_struct_ood_te,  struct_ood_names)

with open(SHARED / "base_features.pkl", "wb") as f:
    pickle.dump(base_bundle, f)
with open(SHARED / "ood_features.pkl", "wb") as f:
    pickle.dump(ood_bundle, f)

print(f"\nKaydedildi:")
print(f"  base_features.pkl ({(SHARED/'base_features.pkl').stat().st_size/1e6:.1f} MB)")
print(f"  ood_features.pkl  ({(SHARED/'ood_features.pkl').stat().st_size/1e6:.1f} MB)")

# ======================================================================
# 3) SANITY BASELINE
# ======================================================================
print("\n" + "="*70)
print("SANITY BASELINE — Q TAHMİNİ")
print("="*70)

# Birleşik feature: TF-IDF + SPECTER2 + yapısal-base + yazar
# (TF-IDF sparse, diğerleri dense → hepsini sparse hstack ile)
def combine(tfidf, specter, struct, auth):
    return hstack([
        tfidf,
        csr_matrix(specter),
        csr_matrix(struct),
        csr_matrix(auth),
    ]).tocsr()

X_full_tr = combine(X_tfidf_tr, X_sp_tr, X_struct_base_tr, X_auth_tr)
X_full_te = combine(X_tfidf_te, X_sp_te, X_struct_base_te, X_auth_te)
print(f"Birleşik feature matris: {X_full_tr.shape}")

# Baseline 1: dummy (random by class freq)
dummy = DummyClassifier(strategy="stratified", random_state=42).fit(X_full_tr, y_tr)
y_pred_dummy = dummy.predict(X_full_te)
print(f"\n[1] Dummy (stratified): "
      f"acc={accuracy_score(y_te, y_pred_dummy):.4f}, "
      f"macro_f1={f1_score(y_te, y_pred_dummy, average='macro'):.4f}")

# Baseline 2: main_cat majority Q
# Train'de her main_cat'in en sık Q'sunu bul, test'e uygula
maincat_to_q = tr.groupby("main_cat")["modded_Q_category"].agg(lambda x: x.value_counts().idxmax())
global_majority = tr["modded_Q_category"].value_counts().idxmax()
y_pred_maincat = te["main_cat"].map(maincat_to_q).fillna(global_majority).values
y_pred_maincat_enc = y_le.transform(y_pred_maincat)
print(f"\n[2] main_cat majority Q: "
      f"acc={accuracy_score(y_te, y_pred_maincat_enc):.4f}, "
      f"macro_f1={f1_score(y_te, y_pred_maincat_enc, average='macro'):.4f}")

# Baseline 3: LR + birleşik feature, train CV
print(f"\n[3] LR + birleşik feature (train 5-fold CV)")
lr = LogisticRegression(max_iter=2000, solver="liblinear")
cv_scores = cross_val_score(lr, X_full_tr, y_tr, cv=5, scoring="f1_macro", n_jobs=-1)
print(f"   train CV macro F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Baseline 4: LR + birleşik feature, fit train, test Bahçeşehir
print(f"\n[4] LR + birleşik feature (Bahçeşehir OOD test)")
lr.fit(X_full_tr, y_tr)
y_pred_lr = lr.predict(X_full_te)
print(f"   OOD test acc: {accuracy_score(y_te, y_pred_lr):.4f}")
print(f"   OOD test macro F1: {f1_score(y_te, y_pred_lr, average='macro'):.4f}")
print(f"\n   Sınıf bazlı detay:")
print(classification_report(y_te, y_pred_lr, target_names=y_le.classes_, digits=3))

# Özet karşılaştırma tablosu
print("\n" + "="*70)
print("BASELINE ÖZET (Bahçeşehir OOD test)")
print("="*70)
summary = pd.DataFrame([
    {"baseline": "Dummy (stratified)",     "acc": accuracy_score(y_te, y_pred_dummy),       "macro_f1": f1_score(y_te, y_pred_dummy, average='macro')},
    {"baseline": "main_cat majority Q",    "acc": accuracy_score(y_te, y_pred_maincat_enc), "macro_f1": f1_score(y_te, y_pred_maincat_enc, average='macro')},
    {"baseline": "LR + birleşik feature",  "acc": accuracy_score(y_te, y_pred_lr),          "macro_f1": f1_score(y_te, y_pred_lr, average='macro')},
])
print(summary.round(4).to_string(index=False))


Yapısal (base, university YOK): (21303, 269)
Univ encoding sütunları: ['univ_q1_rate', 'univ_q2_rate', 'univ_q3_rate', 'univ_q4_rate', 'univ_n_papers_log']
Bilinen üniversite sayısı: 16
Global fill (Bahçeşehir gibi unseen için): {'univ_q1_rate': 0.2470901290022633, 'univ_q2_rate': 0.32078111068602383, 'univ_q3_rate': 0.21402101801008372, 'univ_q4_rate': 0.2181077423016291, 'univ_n_papers_log': 6.490790786089111}
Univ encoded: train (21303, 5), test (807, 5)
Test (Bahçeşehir, hep aynı değer): [0.24709013 0.32078111 0.21402102 0.21810774 6.49079079]

Kaydedildi:
  base_features.pkl (145.8 MB)
  ood_features.pkl  (146.7 MB)

SANITY BASELINE — Q TAHMİNİ
Birleşik feature matris: (21303, 51042)

[1] Dummy (stratified): acc=0.2119, macro_f1=0.2135

[2] main_cat majority Q: acc=0.4672, macro_f1=0.4529

[3] LR + birleşik feature (train 5-fold CV)
   train CV macro F1: 0.4379 ± 0.0167

[4] LR + birleşik feature (Bahçeşehir OOD test)
   OOD test acc: 0.5626
   OOD test macro F1: 0.5469

   Sınıf 